<a href="https://colab.research.google.com/github/munnurumahesh03-coder/nothing/blob/main/notebookc215461785.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

maheshmunnuru_01_eda_and_baseline_ipynb_path = kagglehub.notebook_output_download('maheshmunnuru/01-eda-and-baseline-ipynb')

print('Data source import complete.')


In [ ]:
# GPU acceleration
import cudf         # GPU-accelerated dataframes
import cupy         # GPU-accelerated arrays
from cuml.preprocessing import TargetEncoder  # GPU target encoding

# Standard ML
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold

In [ ]:
import time
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

import cupy as cp
from cuml.preprocessing import TargetEncoder as CuMLTargetEncoder

warnings.filterwarnings('ignore')

In [ ]:
# Load data directly as pandas (lightweight initial load)
train_pd = pd.read_csv('/kaggle/input/01-eda-and-baseline-ipynb/train_cleaned.csv')
test_pd = pd.read_csv('/kaggle/input/01-eda-and-baseline-ipynb/test_cleaned.csv')

print(f'Train Shape: {train_pd.shape}')
print(f'Test Shape: {test_pd.shape}')

TARGET = 'loan_paid_back'

# ==============================================================================
# REVISED FEATURE LIST DEFINITION (Your excellent suggestion)
# ==============================================================================

# 1. Define the original categorical features explicitly.
CATS = [
    'gender',
    'marital_status',
    'education_level',
    'employment_status',
    'loan_purpose',
    'grade_subgrade'
]

# 2. Define the original numerical features explicitly.
# This is "everything else" minus the ID, target, and categoricals.
NUMS = [
    col for col in train_pd.columns
    if col not in ['id', TARGET] and col not in CATS
]

# 3. 'BASE' now logically represents the original feature set.
BASE = CATS + NUMS

print("\n" + "="*50)
print("Feature Lists Defined:")
print(f"  - {len(CATS)} Categorical Features: {CATS}")
print(f"  - {len(NUMS)} Numerical Features: {NUMS}")
print(f"  - {len(BASE)} Total Base Features")
print("="*50)


Train Shape: (593994, 12)
Test Shape: (254569, 12)

Feature Lists Defined:
  - 6 Categorical Features: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
  - 5 Numerical Features: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
  - 11 Total Base Features


In [ ]:
print("\n" + "=" * 80)
print("Creating 2-Way and 3-Way CATEGORICAL Interaction Features")
print("=" * 80)

start_time = time.time()
INTER_FEATURES = [] # Use a more descriptive name

# --- 1. Generate 2-Way Interactions (as before) ---
print(f"Creating 2-way interactions from {len(CATS)} categorical features...")
two_way_count = 0
for col1, col2 in combinations(BASE, 2):
    new_col_name = f'{col1}_{col2}'
    INTER_FEATURES.append(new_col_name)

    for df in [train_pd, test_pd]:
        df[new_col_name] = df[col1].astype(str) + '_' + df[col2].astype(str)
    two_way_count += 1

print(f"  - ✓ Created {two_way_count} 2-way interaction features.")


# --- 2. Generate 3-Way Interactions (your new idea) ---
print(f"\nCreating 3-way interactions from {len(CATS)} categorical features...")
three_way_count = 0
for col1, col2, col3 in combinations(CATS, 3):
    new_col_name = f'{col1}_{col2}_{col3}'
    INTER_FEATURES.append(new_col_name)

    for df in [train_pd, test_pd]:
        # Combine the three columns into a single string feature
        df[new_col_name] = df[col1].astype(str) + '_' + df[col2].astype(str) + '_' + df[col3].astype(str)
    three_way_count += 1

print(f"  - ✓ Created {three_way_count} 3-way interaction features.")


# --- Final Summary ---
elapsed = time.time() - start_time
print("\n" + "=" * 80)
print(f'✓ Created a total of {len(INTER_FEATURES)} interaction features in {elapsed:.2f}s')
print("=" * 80)

# It's good practice to update the main feature list name to avoid confusion
INTER = INTER_FEATURES



Creating 2-Way and 3-Way CATEGORICAL Interaction Features
Creating 2-way interactions from 6 categorical features...
  - ✓ Created 55 2-way interaction features.

Creating 3-way interactions from 6 categorical features...
  - ✓ Created 20 3-way interaction features.

✓ Created a total of 75 interaction features in 36.86s


In [ ]:
# ==============================================================================
# NEW CELL: Numerical Binning (CORRECTED & NaN-SAFE)
# ==============================================================================
print("\n" + "=" * 80)
print("Creating Binned Features (Simple, Direct, and NaN-Safe)")
print("=" * 80)

BINS = []
QUANTILES = [5, 10, 15]

for col in NUMS:
    for q in QUANTILES:
        new_col_name = f"{col}_bin{q}"

        # --- THE FIX: Apply qcut and immediately fill any resulting NaNs ---
        # This ensures the new column has the same length as the dataframe.
        train_pd[new_col_name] = pd.qcut(train_pd[col], q=q, labels=False, duplicates="drop").fillna(-1)
        test_pd[new_col_name] = pd.qcut(test_pd[col], q=q, labels=False, duplicates="drop").fillna(-1)

        BINS.append(new_col_name)

# Add the new feature names to the INTER list for target encoding
BASE.extend(BINS)
print(f"✓ Created and added {len(BINS)} new NaN-safe binned features to the INTER list.")
print(f"  New total count of INTER features: {len(INTER)}")



Creating Binned Features (Simple, Direct, and NaN-Safe)
✓ Created and added 15 new NaN-safe binned features to the INTER list.
  New total count of INTER features: 75


In [ ]:
print("\n" + "=" * 80)
print("Converting to cuDF for GPU Operations")
print("=" * 80)

convert_start = time.time()
train_cudf = cudf.DataFrame.from_pandas(train_pd)
test_cudf = cudf.DataFrame.from_pandas(test_pd)
convert_time = time.time() - convert_start
print(f'✓ Converted to cuDF in {convert_time:.2f}s')


Converting to cuDF for GPU Operations
✓ Converted to cuDF in 6.79s


In [ ]:
# ==============================================================================
# NEW CELL: Leak-Proof Target Aggregations (Replaces the orig_df group features)
# This creates the MEAN and STD features you want, but safely.
# ==============================================================================
print("\n" + "=" * 80)
print("Creating Leak-Proof Target Aggregations (GPU)")
print("=" * 80)

fe_start = time.time()

# We will create aggregations for the original categoricals and the new interactions
FEATURES_TO_AGGREGATE = CATS
AGG_FEATURES = []

# We need a copy of the training data in pandas for the CV split
train_pd_for_split = train_cudf.to_pandas()
y_pd_for_split = train_pd_for_split[TARGET]

# Set up the CV loop
N_CV_SPLITS = 5
skf = StratifiedKFold(n_splits=N_CV_SPLITS, shuffle=True, random_state=42)

# Initialize new columns in our main cuDF dataframe with NaNs
for col in FEATURES_TO_AGGREGATE:
    train_cudf[f'TE_MEAN_{col}'] = cp.nan
    train_cudf[f'TE_STD_{col}'] = cp.nan
    AGG_FEATURES.extend([f'TE_MEAN_{col}', f'TE_STD_{col}'])

print(f"  Aggregating {len(FEATURES_TO_AGGREGATE)} features over {N_CV_SPLITS} folds...")

# --- Main CV Loop for Safe Aggregations ---
for fold, (train_idx, val_idx) in enumerate(skf.split(train_pd_for_split, y_pd_for_split), 1):
    print(f"    Processing Fold {fold}/{N_CV_SPLITS}...")

    # Get the training data for this fold (on GPU)
    X_train_fold = train_cudf.iloc[train_idx]

    # Calculate aggregations ONLY on the training part of the fold
    for col in FEATURES_TO_AGGREGATE:
        # Calculate mean and std for the current column
        agg_stats = X_train_fold.groupby(col)[TARGET].agg(['mean', 'std'])

        # Get the validation part of the fold (on GPU)
        X_val_fold = train_cudf.iloc[val_idx]

        # Map the calculated stats to the validation part
        mean_map = X_val_fold[col].map(agg_stats['mean'])
        std_map = X_val_fold[col].map(agg_stats['std'])

        # Assign the mapped values to the correct rows in the main dataframe
        train_cudf[f'TE_MEAN_{col}'].iloc[val_idx] = mean_map
        train_cudf[f'TE_STD_{col}'].iloc[val_idx] = std_map

# --- Final Aggregations for the Test Set ---
# Now, calculate aggregations on the ENTIRE training set to apply to the test set
print("\n  Calculating final aggregations for test set...")
for col in FEATURES_TO_AGGREGATE:
    agg_stats_full = train_cudf.groupby(col)[TARGET].agg(['mean', 'std'])

    # Map to the test set
    test_cudf[f'TE_MEAN_{col}'] = test_cudf[col].map(agg_stats_full['mean'])
    test_cudf[f'TE_STD_{col}'] = test_cudf[col].map(agg_stats_full['std'])

# Fill any remaining NaNs (e.g., from rare categories) with the global mean/std
global_mean = train_cudf[TARGET].mean()
global_std = train_cudf[TARGET].std()
train_cudf.fillna(global_mean, inplace=True) # A simple fill with mean is okay here
test_cudf.fillna(global_mean, inplace=True)

fe_time = time.time() - fe_start
print(f"\n✓ Created {len(AGG_FEATURES)} new aggregation features in {fe_time:.2f}s")



Creating Leak-Proof Target Aggregations (GPU)
  Aggregating 6 features over 5 folds...
    Processing Fold 1/5...
    Processing Fold 2/5...
    Processing Fold 3/5...
    Processing Fold 4/5...
    Processing Fold 5/5...

  Calculating final aggregations for test set...

✓ Created 12 new aggregation features in 11.45s


In [ ]:
# ==============================================================================
# FINAL STEP: Define Feature List and Convert to Pandas
# ==============================================================================

# --- IMPORTANT: Use our new AGG_FEATURES list ---
# The original notebook used 'ORIG'. We are using 'AGG_FEATURES'.
FEATURES = BASE + AGG_FEATURES + INTER

print(f'\n{"=" * 80}')
print(f'Total Features: {len(FEATURES)}')
print(f'  - Base Features: {len(BASE)}')
print(f'  - Our New Aggregation Features: {len(AGG_FEATURES)}') # Changed this line
print(f'  - Interaction Features: {len(INTER)}') # Changed this line
print("=" * 80)

# Convert back to pandas for XGBoost compatibility
print("\n" + "=" * 80)
print("Converting back to pandas for training")
print("=" * 80)
convert_back_start = time.time()

# Ensure all data is on the host (CPU) before converting
train = train_cudf.to_pandas()
test = test_cudf.to_pandas()

convert_back_time = time.time() - convert_back_start
print(f'✓ Converted back to pandas in {convert_back_time:.2f}s')

# Define final training and target variables
X = train[FEATURES]
y = train[TARGET]

print("\n✓ Data is now ready for XGBoost training.")
print(f"  - Training data shape: {X.shape}")
print(f"  - Test data shape: {test[FEATURES].shape}")



Total Features: 113
  - Base Features: 26
  - Our New Aggregation Features: 12
  - Interaction Features: 75

Converting back to pandas for training
✓ Converted back to pandas in 8.91s

✓ Data is now ready for XGBoost training.
  - Training data shape: (593994, 113)
  - Test data shape: (254569, 113)


In [ ]:
N_SPLITS = 5

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'early_stopping_rounds': 100,
    'random_state': 42,
    'n_jobs': -1,
    'tree_method': 'hist',  # GPU acceleration
    'predictor': 'gpu_predictor',  # GPU predictor
    'device': 'cuda',
    'enable_categorical': True,
}

print(f"\n{'=' * 80}")
print("XGBoost Configuration (GPU Accelerated)")
print('=' * 80)
for k, v in params.items():
    print(f"  {k}: {v}")


XGBoost Configuration (GPU Accelerated)
  objective: binary:logistic
  eval_metric: auc
  max_depth: 5
  colsample_bytree: 0.8
  subsample: 0.8
  n_estimators: 10000
  learning_rate: 0.01
  early_stopping_rounds: 100
  random_state: 42
  n_jobs: -1
  tree_method: hist
  predictor: gpu_predictor
  device: cuda
  enable_categorical: True


In [ ]:
print("\n" + "=" * 80)
print("Starting Cross-Validation Training")
print("=" * 80)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

fold_scores = []
fold_times = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f'\n{"─" * 80}')
    print(f'Fold {fold}/{N_SPLITS}')
    print(f'{"─" * 80}')
    fold_start = time.time()

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_test = test[FEATURES].copy()

    # ========================================================================
    # GPU-Accelerated Target Encoding with cuML
    # ========================================================================
    print("Target Encoding with cuML (GPU)...")
    te_start = time.time()

    # Convert target to cuDF
    y_train_cudf = cudf.Series(y_train.values)

    # Initialize storage for encoded features
    X_train_encoded_dict = {}
    X_val_encoded_dict = {}
    X_test_encoded_dict = {}

    # Process each column individually
    print(f"  Encoding {len(INTER)} interaction features...")
    for idx, col in enumerate(INTER, 1):
        # Convert single column to cuDF
        X_train_col = cudf.Series(X_train[col].values.flatten())
        X_val_col = cudf.Series(X_val[col].values.flatten())
        X_test_col = cudf.Series(X_test[col].values.flatten())

        # Initialize cuML TargetEncoder
        TE = CuMLTargetEncoder(
            n_folds=5,
            smooth=1.0,
            split_method='interleaved',
            output_type='numpy'
        )

        # Fit and transform
        train_encoded = TE.fit_transform(X_train_col, y_train_cudf)
        val_encoded = TE.transform(X_val_col)
        test_encoded = TE.transform(X_test_col)

        # Store encoded values
        encoded_col_name = f'TE_{col}_mean'
        X_train_encoded_dict[encoded_col_name] = train_encoded.flatten()
        X_val_encoded_dict[encoded_col_name] = val_encoded.flatten()
        X_test_encoded_dict[encoded_col_name] = test_encoded.flatten()

        if idx % 10 == 0:
            print(f"    Encoded {idx}/{len(INTER)} columns...")

    # Convert to pandas DataFrames
    X_train_encoded_df = pd.DataFrame(X_train_encoded_dict, index=X_train.index)
    X_val_encoded_df = pd.DataFrame(X_val_encoded_dict, index=X_val.index)
    X_test_encoded_df = pd.DataFrame(X_test_encoded_dict, index=X_test.index)

    # Drop original INTER columns and add encoded features
    X_train_final = X_train.drop(columns=INTER)
    X_val_final = X_val.drop(columns=INTER)
    X_test_final = X_test.drop(columns=INTER)

    X_train_final = pd.concat([X_train_final, X_train_encoded_df], axis=1)
    X_val_final = pd.concat([X_val_final, X_val_encoded_df], axis=1)
    X_test_final = pd.concat([X_test_final, X_test_encoded_df], axis=1)

    te_time = time.time() - te_start
    print(f'  ✓ Target encoding completed in {te_time:.2f}s')

    # Convert categorical columns
    for col in CATS:
        X_train_final[col] = X_train_final[col].astype('category')
        X_val_final[col] = X_val_final[col].astype('category')
        X_test_final[col] = X_test_final[col].astype('category')

    # ========================================================================
    # Train XGBoost Model (GPU)
    # ========================================================================
    print("Training XGBoost (GPU)...")
    model_start = time.time()

    model = XGBClassifier(**params)
    model.fit(X_train_final, y_train,
              eval_set=[(X_val_final, y_val)],
              verbose=1000)

    model_time = time.time() - model_start
    print(f'  ✓ Training completed in {model_time:.2f}s')

    # ========================================================================
    # Prediction and Evaluation
    # ========================================================================
    val_preds = model.predict_proba(X_val_final)[:, 1]
    oof_preds[val_idx] = val_preds

    fold_score = roc_auc_score(y_val, val_preds)
    fold_scores.append(fold_score)

    test_preds += model.predict_proba(X_test_final)[:, 1] / N_SPLITS

    fold_time = time.time() - fold_start
    fold_times.append(fold_time)

    print(f'\n  Fold {fold} Results:')
    print(f'    AUC: {fold_score:.4f}')
    print(f'    Time: {fold_time:.2f}s (TE: {te_time:.2f}s, Model: {model_time:.2f}s)')


Starting Cross-Validation Training

────────────────────────────────────────────────────────────────────────────────
Fold 1/5
────────────────────────────────────────────────────────────────────────────────
Target Encoding with cuML (GPU)...
  Encoding 75 interaction features...
    Encoded 10/75 columns...
    Encoded 20/75 columns...
    Encoded 30/75 columns...
    Encoded 40/75 columns...
    Encoded 50/75 columns...
    Encoded 60/75 columns...
    Encoded 70/75 columns...
  ✓ Target encoding completed in 17.53s
Training XGBoost (GPU)...
[0]	validation_0-auc:0.90884
[1000]	validation_0-auc:0.92560
[2000]	validation_0-auc:0.92597
[2380]	validation_0-auc:0.92599
  ✓ Training completed in 41.08s

  Fold 1 Results:
    AUC: 0.9260
    Time: 63.06s (TE: 17.53s, Model: 41.08s)

────────────────────────────────────────────────────────────────────────────────
Fold 2/5
────────────────────────────────────────────────────────────────────────────────
Target Encoding with cuML (GPU)...
  Enc

In [ ]:
overall_auc = roc_auc_score(y, oof_preds)

print('\n' + '=' * 80)
print('Final Results')
print('=' * 80)
print(f'Overall OOF AUC: {overall_auc:.4f}')
print(f'Mean Fold AUC: {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})')
print(f'Total Training Time: {sum(fold_times):.2f}s')
print(f'Average Time per Fold: {np.mean(fold_times):.2f}s')
print('=' * 80)

print('\nFold Scores:')
for i, score in enumerate(fold_scores, 1):
    print(f'  Fold {i}: {score:.4f} ({fold_times[i - 1]:.2f}s)')


Final Results
Overall OOF AUC: 0.9258
Mean Fold AUC: 0.9258 (±0.0008)
Total Training Time: 301.27s
Average Time per Fold: 60.25s

Fold Scores:
  Fold 1: 0.9260 (63.06s)
  Fold 2: 0.9270 (61.02s)
  Fold 3: 0.9246 (60.74s)
  Fold 4: 0.9259 (59.56s)
  Fold 5: 0.9256 (56.90s)


In [ ]:
print(f"\n{'=' * 80}")
print("Creating OOF and Submission Files...")
print('=' * 80)

# --- 1. Create the OOF predictions file (with synthetic IDs) ---
# Your idea: create synthetic IDs from scratch.
synthetic_train_ids = np.arange(len(oof_preds))

# Create the DataFrame and save it.
oof_df = pd.DataFrame({'id': synthetic_train_ids, TARGET: oof_preds})
oof_df.to_csv('oof_predictions.csv', index=False)

print("✓ 'oof_predictions.csv' created successfully with synthetic IDs.")


# --- 2. Create the final submission file (with real test IDs) ---
# We use 'test_pd' which has the real 'id' column.
submission_df = pd.DataFrame({'id': test_pd['id'], TARGET: test_preds})
submission_df.to_csv('submission.csv', index=False)

print("✓ 'submission.csv' created successfully with real test IDs.")



Creating OOF and Submission Files...
✓ 'oof_predictions.csv' created successfully with synthetic IDs.
✓ 'submission.csv' created successfully with real test IDs.
